In [ ]:
# Install required dependencies for Google Colab
!pip install git+https://github.com/elkins-lab/synth-saxs.git biotite matplotlib py3Dmol ipywidgets

# Intrinsically Disordered Proteins (IDPs) vs. Folded Proteins

Small-Angle X-ray Scattering (SAXS) is uniquely suited to study **Intrinsically Disordered Proteins (IDPs)** because it naturally measures the protein in a solution state, without forcing it into a crystal lattice.

In this tutorial, we will use `synth-saxs` to simulate and contrast the scattering profiles of:
1. **Hen Egg-White Lysozyme (PDB: 1AKI)** - A classic, highly rigid, globular folded protein.
2. **p53 Transactivation Domain (PDB: 2L42)** - An NMR ensemble of a highly flexible, intrinsically disordered region.

The signature of disorder is best observed in the **Kratky plot** ($q^2 \cdot I(q)$ vs $q$).
- Folded proteins show a distinct "bell shape" (a prominent peak that returns to near-zero at high $q$).
- IDPs show a rising curve that plateaus or continues to increase at high $q$ (because the protein behaves like a random polymer coil).


In [ ]:
import biotite.database.rcsb as rcsb
import biotite.structure.io as strucio
import matplotlib.pyplot as plt
import py3Dmol

from synth_saxs.engine import calculate_saxs_profile, preprocess_structure

%matplotlib inline

## 1. The Folded Protein (Lysozyme)

In [ ]:
print("Downloading Folded Protein (1AKI)...")
pdb_folded = rcsb.fetch("1AKI", "pdb", target_path="1AKI.pdb")
struct_folded = strucio.load_structure(pdb_folded)

# Coarse-grain to C-alpha atoms for speed
struct_folded = preprocess_structure(struct_folded)
# Coarse-grain to CA atoms
struct_folded = struct_folded[struct_folded.atom_name == "CA"]

# 3D Visualization
viewer = py3Dmol.view(width=400, height=300)
viewer.addModel(open(pdb_folded).read(), "pdb")
viewer.setStyle({"cartoon": {"color": "spectrum"}})
viewer.zoomTo()
viewer.show()

## 2. The Intrinsically Disordered Protein (p53 TAD)
The PDB file for 2L42 contains an ensemble of many different conformations, representing the highly flexible nature of the IDP in solution.

In [ ]:
print("Downloading IDP Ensemble (2L42)...")
pdb_idp = rcsb.fetch("2L42", "pdb", target_path="2L42.pdb")
struct_idp_ensemble = strucio.load_structure(pdb_idp)

print(f"Loaded ensemble with {struct_idp_ensemble.stack_depth()} different conformational models.")

# We will simulate SAXS for the first 10 models and average them
# (In a real scenario, you might average the entire ensemble)
struct_idp_subset = struct_idp_ensemble[:10]
struct_idp_subset = preprocess_structure(struct_idp_subset)
# Coarse-grain to CA atoms on stack
struct_idp_subset = struct_idp_subset[:, struct_idp_subset.atom_name == "CA"]

# 3D Visualization (showing just the first model)
viewer = py3Dmol.view(width=400, height=300)
viewer.addModel(open(pdb_idp).read(), "pdb")
viewer.setStyle({"cartoon": {"color": "orange"}})
viewer.zoomTo()
viewer.show()

## 3. Simulating the SAXS Profiles
We will simulate SAXS for both the folded protein and the IDP ensemble.
*Note: The IDP calculation averages the scattering intensity across all the models in the ensemble.*

In [ ]:
print("Calculating SAXS for Folded Lysozyme...")
q, i_folded = calculate_saxs_profile(
    struct_folded, q_min=0.01, q_max=0.3, n_points=100, include_solvent=True
)

print("Calculating ensemble-averaged SAXS for IDP (p53)...")
q, i_idp = calculate_saxs_profile(
    struct_idp_subset, q_min=0.01, q_max=0.3, n_points=100, include_solvent=True
)
print("Done!")

## 4. The Kratky Plot Analysis
Let's normalize the maximum intensity of both curves to 1.0 just to compare their shapes, and plot them on a Kratky plot.

In [ ]:
# Normalize I(q) to I(0)=1 for shape comparison
i_folded_norm = i_folded / i_folded[0]
i_idp_norm = i_idp / i_idp[0]

kratky_folded = (q**2) * i_folded_norm
kratky_idp = (q**2) * i_idp_norm

plt.figure(figsize=(8, 6))
plt.plot(q, kratky_folded, "b-", lw=3, label="Folded (Lysozyme)")
plt.plot(q, kratky_idp, "r-", lw=3, label="IDP Ensemble (p53)")

plt.title("Kratky Plot: Folded vs. Intrinsically Disordered", fontsize=14)
plt.xlabel(r"$q$ ($\AA^{-1}$)", fontsize=12)
plt.ylabel(r"$q^2 \cdot I(q)$ (Normalized)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

# Annotate the shapes
plt.annotate(
    "Bell Shape\n(Folded Core)",
    xy=(0.12, kratky_folded.max()),
    xytext=(0.15, kratky_folded.max() + 0.005),
    arrowprops={"facecolor": "blue", "shrink": 0.05},
    fontsize=11,
    color="blue",
)

plt.annotate(
    "Rising Plateau\n(Random Coil)",
    xy=(0.25, kratky_idp[-1]),
    xytext=(0.15, kratky_idp[-1] - 0.005),
    arrowprops={"facecolor": "red", "shrink": 0.05},
    fontsize=11,
    color="red",
)

plt.show()

### Conclusion
Notice how the blue curve (Folded) peaks and drops back down, indicating a compact, rigid mass. 
The red curve (IDP), however, lacks a strong peak and instead levels off or rises at higher $q$ values, which is the classic mathematical signature of a flexible, polymer-like chain in solution!